# Free-Boundary PF-Coil Scans with TokaMaker

Issue #67's free-boundary scan workflow: starting from a measured VEST machine
state, PF-coil currents are varied and each resulting free-boundary equilibrium
is solved with **TokaMaker** and classified (`vaft.code.free_boundary_scan`).

    controls ── commanded coil currents (absolute/offset/scale per coil set)
             ── one TokaMaker instance, continuation warm starts
             ── per-case g-file + case_manifest.json (+ scan_manifest.json, resume)
             ── topology classification: limited / near-null / SN / DN,
                X-points, dRsep, limiter contact, discontinuity flags

Every case records the *commanded* and the solver-*materialized* currents, the
held targets (`Ip`, profile shape, optional axis position) and the achieved
global quantities. Failed and non-converged cases stay visible — the scan never
bridges a gap. Solver cells are skipped when OpenFUSIONToolkit is unavailable.

In [ ]:
import json
import time as _time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import vaft
from vaft.code import tokamaker
from vaft.code.tokamaker import TokaMakerConfig, free_boundary_scan

WORKDIR = Path("_tokamaker_run") / "free_boundary"
WORKDIR.mkdir(parents=True, exist_ok=True)

try:
    tokamaker._oft.import_oft()
    OFT_AVAILABLE = True
except ImportError as exc:
    OFT_AVAILABLE = False
    print(exc)
print("OpenFUSIONToolkit available:", OFT_AVAILABLE)

## 1. Load the baseline machine state

The scan baseline is the measured shot: coil currents from `pf_active` at the
chosen time, the Ip target from the equilibrium/magnetics IDS, and the
canonical VEST limiter. The full 10-coil `pf_active` set is needed — the
packaged offline sample currently carries only PF1–PF5, so the database load
is preferred and the outboard-coil sections degrade gracefully without it.

In [ ]:
SHOT, TIME = 39915, 0.325
try:
    ods = vaft.database.load(SHOT)
    print(f"loaded shot {SHOT} via vaft.database.load")
except Exception as e:
    ods = vaft.omas.load(vaft.data.sample(SHOT))
    print(f"vaft.database.load unavailable ({type(e).__name__}); using the packaged sample")

from vaft.machine_mapping.wall import wall as canonical_wall
canonical_wall(ods)

N_COILS = len(ods["pf_active.coil"])
FULL_MACHINE = N_COILS == 10
print(f"pf_active coils: {N_COILS} (full machine: {FULL_MACHINE})")

## 2. A PF-coil offset scan

Five equilibria along a PF6 offset axis (the strongest outboard shaping coil at
this time slice), solved with continuation from a shared TokaMaker instance.
`hold=("ip", "profile_shape")` keeps the plasma-current target and the
profile shape fixed; everything else responds to the coil change.

In [ ]:
COIL = "PF6" if FULL_MACHINE else "PF5"
if OFT_AVAILABLE:
    cfg = TokaMakerConfig(shot=SHOT, time=TIME, workdir=WORKDIR)
    scan_a = free_boundary_scan(
        ods,
        controls={COIL: {"offset_A": [-800.0, -400.0, 0.0, 400.0, 800.0]}},
        config=cfg, workdir=WORKDIR / "coil_offset",
    )
    t0 = _time.time()
    res_a = scan_a.run(resume=True)
    print(f"{len(res_a.succeeded)}/{len(res_a.cases)} cases in {_time.time()-t0:.0f} s")
    for c in res_a.cases:
        t = c.topology or {}
        contact = (t.get("limiter_contact") or {}).get("distance")
        print(f"  {c.case_id:34s} {c.status.value:12s} "
              f"Ip={c.achieved.get('Ip', float('nan'))/1e3:6.2f} kA  "
              f"topo={t.get('topology','-'):10s} wall gap={contact}")
else:
    print("OpenFUSIONToolkit not available - skipping")

## 3. Machine response along the axis

Commanded vs materialized currents, the LCFS family, and the limiter-contact
clearance as functions of the control.

In [ ]:
if OFT_AVAILABLE:
    from vaft.data.eqdsk import read_geqdsk

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    offsets, gaps, kappas = [], [], []
    colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(res_a.cases)))
    lim_r = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.r"])
    lim_z = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.z"])
    axes[0].plot(np.r_[lim_r, lim_r[0]], np.r_[lim_z, lim_z[0]], "k-", lw=1)
    for c, color in zip(res_a.cases, colors):
        if not c.ok:
            continue
        offset = list(c.requested[COIL].values())[0]
        offsets.append(offset)
        gaps.append((c.topology.get("limiter_contact") or {}).get("distance"))
        kappas.append(c.achieved.get("kappa"))
        g = read_geqdsk(c.gfile)
        axes[0].plot(g["RBBBS"], g["ZBBBS"], color=color, lw=0.9,
                     label=f"{offset:+.0f} A")
    axes[0].set_xlabel("R [m]"); axes[0].set_ylabel("Z [m]")
    axes[0].set_aspect("equal"); axes[0].legend(fontsize=7, title=f"{COIL} offset")
    axes[1].plot(offsets, np.asarray(gaps) * 1e3, "o-")
    axes[1].set_xlabel(f"{COIL} offset [A]"); axes[1].set_ylabel("LCFS-wall clearance [mm]")
    axes[2].plot(offsets, kappas, "s-")
    axes[2].set_xlabel(f"{COIL} offset [A]"); axes[2].set_ylabel("elongation")
    fig.tight_layout(); plt.show()
else:
    print("OpenFUSIONToolkit not available - skipping")

## 4. Topology-transition trajectory: limited -> near-null

An up/down-asymmetric drive needs independent coil halves:
`split_coils=("PF6",)` makes `PF6_U`/`PF6_L` separate coil sets, and a
Vertical Stability Coil pair (`vsc_coil="PF9"`) plus `v0_target` holds the
magnetic axis while the lower field is reshaped. Marching `PF6_L` from the
measured −1430 A through zero to +900 A (current co-directed with Ip) pulls a
lower null toward the plasma: the classification walks `limited -> near_null`,
with the approach margin |psi_N − 1| of the incoming null as the transition
marker. Beyond ~+1800 A the fixed-point solve stops converging before the null
activates — those cases are recorded as `not_converged`, never interpolated.

In [ ]:
if OFT_AVAILABLE and FULL_MACHINE:
    cfg_split = TokaMakerConfig(shot=SHOT, time=TIME, workdir=WORKDIR,
                                split_coils=("PF6",), vsc_coil="PF9", v0_target=0.0)
    trajectory = [-1430.0, -700.0, 0.0, 700.0, 900.0, 1300.0, 1700.0, 1900.0]
    scan_b = free_boundary_scan(
        ods, mode="zip",
        controls={"PF6_L": {"absolute_A": trajectory}},
        config=cfg_split, workdir=WORKDIR / "transition",
        hold=("ip", "profile_shape", "axis_position"),
        refine_on_failure=1, active_tolerance=5.0e-3,
    )
    res_b = scan_b.run(resume=True)
    print(f"{len(res_b.succeeded)}/{len(res_b.cases)} converged")
    for c in res_b.cases:
        t = c.topology or {}
        margin = t.get("null_margin")
        print(f"  PF6_L={list(c.requested['PF6_L'].values())[0]:+7.0f} A  "
              f"{c.status.value:13s} topo={t.get('topology','-'):12s} "
              f"null margin={margin if margin is None else round(margin, 4)} "
              f"solver diverted={c.solver_diverted}")
else:
    print("needs OFT and the full 10-coil machine state - skipping")

In [ ]:
if OFT_AVAILABLE and FULL_MACHINE:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    xs, margins, statuses = [], [], []
    for c in res_b.cases:
        x = list(c.requested["PF6_L"].values())[0]
        xs.append(x)
        statuses.append(c.status.value)
        margins.append((c.topology or {}).get("null_margin"))
    ok = [i for i, s in enumerate(statuses) if s == "succeeded"]
    bad = [i for i, s in enumerate(statuses) if s != "succeeded"]
    axes[0].plot([xs[i] for i in ok],
                 [margins[i] if margins[i] is not None else np.nan for i in ok],
                 "o-", label="approach margin |psi_N-1|")
    for i in bad:                       # failures stay visible, never bridged
        axes[0].axvline(xs[i], color="tab:red", alpha=0.3, lw=6)
    axes[0].axhline(5e-3, color="k", ls=":", label="active tolerance")
    axes[0].set_xlabel("PF6_L [A]"); axes[0].set_ylabel("null approach margin")
    axes[0].legend(fontsize=8)
    axes[0].set_title("red bands: not_converged cases")

    # incoming-null track from the topology reports
    for c in res_b.cases:
        for xp in (c.topology or {}).get("x_points", []):
            if xp["psi_n"] >= 0.99:
                axes[1].plot(xp["r"], xp["z"], "v", color="tab:blue")
    lim_r = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.r"])
    lim_z = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.z"])
    axes[1].plot(np.r_[lim_r, lim_r[0]], np.r_[lim_z, lim_z[0]], "k-", lw=1)
    axes[1].set_xlabel("R [m]"); axes[1].set_ylabel("Z [m]"); axes[1].set_aspect("equal")
    axes[1].set_title("approaching-null positions")
    fig.tight_layout(); plt.show()
else:
    print("skipping")

## 5. Manifests, resume, and provenance

Each case directory carries a `case_manifest.json` (status, commanded and
materialized currents, held/achieved quantities, topology report, discontinuity
flags, refinement history) keyed by a config hash; `scan_manifest.json`
summarizes the scan. Re-running with `resume=True` reloads succeeded cases
without solving — the cell above is instant on a second execution.

In [ ]:
if OFT_AVAILABLE:
    payload = json.loads((WORKDIR / "coil_offset" / "scan_manifest.json").read_text())
    print("scan manifest:", {k: payload[k] for k in ("schema_version", "solver", "mode", "hold")})
    case0 = res_a.cases[0]
    case_payload = json.loads(case0.manifest.read_text())
    print("case keys:", sorted(case_payload))
    print("commanded :", {k: round(v, 1) for k, v in case_payload["commanded_currents_A"].items()})
    print("achieved Ip:", round(case_payload["achieved"]["Ip"], 1), "A "
          "(held target:", round(case_payload["held"]["ip"], 1), "A)")
else:
    print("OpenFUSIONToolkit not available - skipping")

## Notes / open tasks

- **Reaching a true single-null on this shot.** The measured 0.325 s state is
  deeply limited; driving `PF6_L` alone stalls (non-convergence) before the
  incoming null activates. A designed multi-coil current set — TokaMaker's
  isoflux/inverse mode, deliberately out of scope here — is the follow-up
  route to full LSN/USN/DN families (issue #67 continuation).
- **dRsep** is reported per case whenever both an upper and a lower separatrix
  candidate exist; on this limited shot it stays undefined.
- **Offline sample.** The packaged 39915 sample currently lacks PF6–PF10;
  full-machine sections need the database connection.
- The TES backend and the solver-neutral contract shared with the CHEASE scan
  framework (#66) are deferred until both implementations are mature.